# COCOMO-Extended — Pilot data-collection notebook (version 2)

**Changes from version 1, based on the first pilot:** quarterly windows; bulk commits (over 2,000 source lines) left out of size; DevOps maturity measured from git (CI workflows, test steps, releases) because GitHub keeps CI-run history for only about a year; an activity screen so inactive repositories are not chosen; AI repositories must have adopted agents by September 2025; AI adoption defined by any AI signal; results saved to a new folder (`pilot_v2`).

This notebook builds the pilot dataset for the paper: **20 public GitHub repositories** (10 that use AI coding agents, 10 matched repositories that do not), measured in **6-month windows from 2021 to mid-2026**.

For every repository-window it computes **effort** (full-time-equivalent developer-months), **size** (KSLOC added/modified) and the **six drivers** (DVOP, AIAD, MSVC, CLAU, MTEX, SECU), each as a 0–1 score and as a COCOMO level (VL = −2 … XH = 3).

**How to run**
1. Create a GitHub token: GitHub → Settings → Developer settings → Personal access tokens → *Fine-grained token*, read-only access to public repositories is enough.
2. In Colab, click the key icon (Secrets) on the left, add a secret named `GITHUB_TOKEN` with the token, and switch on notebook access. (Or paste it when asked.)
3. Run the cells from top to bottom (*Runtime → Run all*). The pilot takes roughly 1–3 hours.
4. Results are saved to Google Drive in `MyDrive/cocomo_extended/pilot/`. If Colab disconnects, run everything again: finished repositories are skipped automatically.

Everything is downloaded automatically: the AIDev dataset from Hugging Face, repository histories with `git clone`, pull-request, release and CI counts from the GitHub API, and security scores from the OpenSSF Scorecard API.

## 1. Setup

In [ ]:
!pip -q install huggingface_hub pyarrow fsspec statsmodels
# Save results to Google Drive if possible; otherwise to the Colab disk (download them at the end).
USE_DRIVE = True
SAVE_ROOT = "/content/cocomo_extended"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_ROOT = "/content/drive/MyDrive/cocomo_extended"
    except Exception as e:
        print("Google Drive could not be mounted:", e)
        print("Continuing WITHOUT Drive. Results go to /content/cocomo_extended and are lost when the "
              "session ends - run the download cell at the end. To retry Drive, run this cell again "
              "and tick 'Select all' on the permissions screen.")
print("Results folder:", SAVE_ROOT)

In [ ]:
# GitHub token: read from Colab Secrets, or ask for it
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    from getpass import getpass
    GITHUB_TOKEN = getpass('Paste your GitHub token: ')
print('Token loaded:', bool(GITHUB_TOKEN))

## 2. Settings
Change values here only. Every cut-off used to turn indicators into scores is listed in `FULL_SCORE_AT` and `LEVEL_CUTS`; **fix them before the full run and record them in the OSF pre-registration**.

`FT_COMMITS_PER_MONTH` is the commit threshold above which a developer counts as one full-time developer in that month (Robles et al., EMSE 2022). Set it from that paper; 10 is a placeholder to check.

In [ ]:
CFG = dict(
    # study period and windows
    START="2021-01-01", END="2026-07-01", WINDOW_MONTHS=3,   # quarterly: more periods after AI adoption
    # effort model (Robles et al. 2022) - CHECK THIS VALUE AGAINST THE PAPER
    FT_COMMITS_PER_MONTH=10,
    # inclusion rules
    CORE_DEV_COMMITS=5, MIN_CORE_DEVS=3, MIN_STARS=100, MIN_KSLOC=1.0,
    BULK_COMMIT_LINES=2000,    # commits adding more source lines than this are left out of size
    LANGS=["Python", "JavaScript", "TypeScript", "Java", "Go", "C#", "Rust"],
    CREATED_BEFORE="2022-01-01", ACTIVE_AFTER="2026-05-01", MAX_REPO_MB=1500,
    ACTIVE_SINCE="2025-01-01", MIN_RECENT_COMMITS=300,   # activity screen before cloning
    # pilot sample
    N_AI=10, MIN_AGENT_PRS=5, ADOPTED_BY="2025-09-30",   # leaves >= 3 quarters after adoption
    MAX_CANDIDATES_CHECKED=300, SEED=42,
    # value at which each indicator counts as fully 'on' (score = 1)
    FULL_SCORE_AT=dict(n_ci_workflows=5, ci_runs_per_week=20, releases_per_month=2, agent_pr_share=0.30,
                       ai_commit_share=0.30, n_services=50, iac_change_share=0.10,
                       tenure_months=72),
    # score -> COCOMO level: <0.10 VL, <0.30 L, <0.50 N, <0.70 H, <0.90 VH, else XH
    LEVEL_CUTS=[0.10, 0.30, 0.50, 0.70, 0.90],
    # where files go
    OUT_DIR=SAVE_ROOT + "/pilot_v2",   # new folder, so version-1 results are not reused
    CLONE_DIR="/content/clones",
    AIDEV_VERSION="v5",
)
import os
os.makedirs(CFG["OUT_DIR"] + "/repos", exist_ok=True)
os.makedirs(CFG["CLONE_DIR"], exist_ok=True)

## 3. Measurement functions
These read a local git clone. Nothing to change here.

In [ ]:
# ===================== MEASUREMENT FUNCTIONS (git-based) =====================
# Everything in this cell works from a local git clone; no API calls.
import os, re, math, json, shutil, subprocess
from collections import defaultdict
import pandas as pd
import numpy as np
import yaml

# ---------- windows ----------
def make_windows(start, end, months):
    """Return list of (label, start_ts, end_ts) half-open windows [start, end)."""
    edges = pd.date_range(start=start, end=end, freq=f"{months}MS", tz="UTC")
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        if months == 6:
            lab = f"{a.year}H{1 if a.month <= 6 else 2}"
        elif months == 3:
            lab = f"{a.year}Q{(a.month - 1) // 3 + 1}"
        else:
            lab = a.strftime("%Y-%m")
        out.append((lab, a, b))
    return out

# ---------- git helpers ----------
def run(cmd, cwd=None, timeout=3600):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                       errors="replace", timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])}... failed: {r.stderr[:300]}")
    return r.stdout

def clone_repo(full_name, dest):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    run(["git", "clone", "--quiet", "--no-checkout", "--single-branch",
         f"https://github.com/{full_name}.git", dest], timeout=5400)

REC, FLD = "\x1e", "\x1f"

def _clean_path(p):
    p = p.strip()
    if "{" in p and "=>" in p:
        p = re.sub(r"\{([^{}]*) => ([^{}]*)\}", r"\2", p)
        p = p.replace("//", "/")
    elif " => " in p:
        p = p.split(" => ")[1]
    return p

def read_commits(repo_dir, since, until):
    """Non-merge commits on the default branch with numstat, between since and until."""
    fmt = f"{REC}%H{FLD}%ae{FLD}%an{FLD}%aI{FLD}%B{FLD}"
    out = run(["git", "log", "--no-merges", "--numstat", f"--format={fmt}",
               f"--since={since}", f"--until={until}"], cwd=repo_dir)
    commits = []
    for rec in out.split(REC)[1:]:
        parts = rec.split(FLD)
        if len(parts) < 6:
            continue
        sha, email, name, date, body, stat = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5]
        files = []
        for line in stat.strip().splitlines():
            cols = line.split("\t")
            if len(cols) != 3:
                continue
            a, d, p = cols
            files.append((int(a) if a.isdigit() else 0, int(d) if d.isdigit() else 0, _clean_path(p)))
        commits.append({"sha": sha, "email": email.lower().strip(), "name": name.strip(),
                        "date": pd.Timestamp(date).tz_convert("UTC"), "body": body, "files": files})
    return commits

def first_commit_dates(repo_dir):
    """First-ever commit date per author email (full history) for team-experience."""
    out = run(["git", "log", "--no-merges", "--format=%ae%x1f%aI", "--reverse"], cwd=repo_dir)
    first = {}
    for line in out.splitlines():
        if FLD not in line:
            continue
        e, d = line.split(FLD)
        e = e.lower().strip()
        if e not in first:
            first[e] = pd.Timestamp(d).tz_convert("UTC")
    return first

def tag_dates(repo_dir):
    out = run(["git", "for-each-ref", "refs/tags", "--format=%(creatordate:iso-strict)"], cwd=repo_dir)
    ds = []
    for line in out.splitlines():
        line = line.strip()
        if line:
            try:
                ds.append(pd.Timestamp(line).tz_convert("UTC"))
            except Exception:
                pass
    return ds

def commit_at(repo_dir, before_ts):
    out = run(["git", "rev-list", "-1", f"--before={before_ts.isoformat()}", "HEAD"], cwd=repo_dir).strip()
    return out or None

def list_files(repo_dir, sha):
    return run(["git", "ls-tree", "-r", "--name-only", sha], cwd=repo_dir).splitlines()

def file_size(repo_dir, sha, path):
    try:
        return int(run(["git", "cat-file", "-s", f"{sha}:{path}"], cwd=repo_dir).strip())
    except Exception:
        return 10**9

def read_file(repo_dir, sha, path, max_bytes=300_000):
    if file_size(repo_dir, sha, path) > max_bytes:
        return ""
    try:
        return run(["git", "show", f"{sha}:{path}"], cwd=repo_dir, timeout=120)
    except Exception:
        return ""

# ---------- classification of authors ----------
BOT_RE = re.compile(r"\[bot\]|(^|[^a-z])bot@|dependabot|renovate|github-actions|pre-commit-ci|codecov|"
                    r"greenkeeper|snyk-bot|allcontributors|semantic-release|mergify|imgbot|"
                    r"weblate|transifex|crowdin|azure-pipelines|travis", re.I)
AI_RE = re.compile(r"copilot|devin-ai|noreply@anthropic\.com|(^|[^a-z])claude($|[^a-z])|"
                   r"chatgpt-codex|openai-codex|codex\[bot\]|cursoragent|cursor\[bot\]|"
                   r"jules\[bot\]|openhands|aider|factory-droid|gemini-code-assist|sweep-ai", re.I)
COAUTHOR_RE = re.compile(r"^co-authored-by:\s*(.+)$", re.I | re.M)

def author_kind(name, email):
    s = f"{name} <{email}>"
    if AI_RE.search(s):
        return "ai"
    if BOT_RE.search(s):
        return "bot"
    return "human"

def has_ai_coauthor(body):
    return any(AI_RE.search(m) for m in COAUTHOR_RE.findall(body or ""))

# ---------- file classification ----------
SRC_EXT = {".py", ".js", ".jsx", ".ts", ".tsx", ".mjs", ".cjs", ".java", ".kt", ".scala", ".go",
           ".cs", ".rs", ".rb", ".php", ".c", ".cc", ".cpp", ".cxx", ".h", ".hpp", ".swift",
           ".m", ".vue", ".svelte", ".dart", ".ex", ".exs", ".clj", ".fs", ".lua"}
EXCL_DIR_RE = re.compile(r"(^|/)(vendor|third_party|thirdparty|node_modules|dist|build|out|target|"
                         r"generated|gen|\.yarn|bower_components|site-packages|external)/", re.I)
EXCL_FILE_RE = re.compile(r"(\.min\.js$|_pb2(_grpc)?\.py$|\.pb\.go$|\.pb\.(h|cc)$|\.g\.dart$|"
                          r"\.generated\.|\.designer\.cs$|bundle\.js$)", re.I)
TEST_RE = re.compile(r"(^|/)(tests?|__tests__|spec|specs|testing|e2e|test_?data|fixtures)/|"
                     r"(^|/)test_[^/]*$|_test\.[a-z]+$|\.(test|spec)\.[a-z]+$|Tests?\.(java|kt|cs)$", re.I)

def file_class(path):
    """'src', 'test' or None (not counted in size)."""
    ext = os.path.splitext(path)[1].lower()
    if ext not in SRC_EXT or EXCL_DIR_RE.search(path) or EXCL_FILE_RE.search(path):
        return None
    return "test" if TEST_RE.search(path) else "src"

CI_RE = re.compile(r"^(\.github/workflows/[^/]+\.ya?ml|\.gitlab-ci\.yml|\.circleci/config\.ya?ml|"
                   r"Jenkinsfile|\.travis\.yml|azure-pipelines\.ya?ml|\.drone\.yml|bitbucket-pipelines\.yml|"
                   r"\.buildkite/.+\.ya?ml)$")
# deployment folders: only at the top level (or one level under deploy-type roots), never inside source code
DEPLOY_ROOT = (r"(ops|tf|cfn|\.?[^/]*(deploy|k8s|kube|helm|chart|kustomiz|infra|terraform|manifest|istio|argocd|"
               r"cloudformation|pulumi|ansible|gitops)[^/]*)")
IAC_RE = re.compile(r"(\.tf$|\.tfvars$|\.bicep$|(^|/)Pulumi\.ya?ml$|(^|/)Chart\.ya?ml$|"
                    r"(^|/)kustomization\.ya?ml$|(^|/)serverless\.ya?ml$|(^|/)cdk\.json$|"
                    r"(^|/)skaffold\.ya?ml$|\.ya?ml$|\.json$)", re.I)
IAC_LOC_RE = re.compile(r"^(" + DEPLOY_ROOT + r"/|[^/]+/" + DEPLOY_ROOT + r"/)", re.I)
IAC_STRONG_RE = re.compile(r"(\.tf$|\.bicep$|(^|/)Pulumi\.ya?ml$|(^|/)cdk\.json$|(^|/)serverless\.ya?ml$|"
                           r"^skaffold\.ya?ml$|^cloudbuild\.ya?ml$|(^|/)Chart\.ya?ml$|(^|/)kustomization\.ya?ml$)", re.I)

def is_iac(path):
    """IaC file in a top-level deployment folder; Terraform/Bicep/Pulumi/CDK/Serverless files anywhere
    outside source, test, example and docs folders."""
    if NONPROD_DIR_RE.search(path):
        return False
    if IAC_STRONG_RE.search(path):
        return True
    return bool(IAC_LOC_RE.search(path)) and path.lower().endswith((".yml", ".yaml", ".json", ".tf", ".tfvars"))

COMPOSE_RE = re.compile(r"(^|/)(docker-)?compose[^/]*\.ya?ml$", re.I)
K8S_DIR_RE = IAC_LOC_RE   # Kubernetes manifests counted only in top-level deployment folders
DOCKERFILE_RE = re.compile(r"(^|/)Dockerfile[^/]*$|\.Dockerfile$", re.I)
NONPROD_DIR_RE = re.compile(r"(^|/)(tests?|testing|testdata|examples?|samples?|docs?|\.devcontainer|benchmarks?|demos?|e2e|"
                            r"fixtures|integration)/", re.I)
PACKAGING_DIR_RE = re.compile(r"(^|/)(dev|docker|ci|scripts?|tools|build|\.github|hack|contrib)(/|$)", re.I)
BUILD_FILE_RE = re.compile(r"(^|/)(package\.json|go\.mod|pyproject\.toml|setup\.py|requirements\.txt|pom\.xml|"
                           r"build\.gradle(\.kts)?|Cargo\.toml|[^/]+\.csproj|Gemfile|composer\.json|mix\.exs)$", re.I)
AI_CFG_RE = re.compile(r"^(\.github/copilot-instructions\.md|\.github/instructions/.+|\.cursorrules|"
                       r"\.cursor/.+|CLAUDE\.md|\.claude/.+|AGENTS\.md|GEMINI\.md|\.aider[^/]*|"
                       r"\.windsurfrules|\.clinerules.*|\.continue/.+)$", re.I)
SECPOL_RE = re.compile(r"^(\.github/|docs/)?SECURITY\.(md|rst|txt)$", re.I)
DEPUPD_RE = re.compile(r"^(\.github/dependabot\.ya?ml|renovate\.json5?|\.renovaterc(\.json)?|"
                       r"\.github/renovate\.json5?)$", re.I)
CODEOWNERS_RE = re.compile(r"^(\.github/|docs/)?CODEOWNERS$")
SAST_RE = re.compile(r"codeql|semgrep|snyk|bandit|gosec|sonar|trivy|grype|brakeman|"
                     r"ossf/scorecard|osv-scanner|zizmor", re.I)
DEPLOY_RE = re.compile(r"terraform\s+(apply|plan)|helm\s+(upgrade|install)|kubectl\s+(apply|rollout|set)|"
                       r"aws\s+(cloudformation|ecs|lambda|s3\s+sync)|gcloud\s+\S*\s*deploy|"
                       r"az\s+(webapp|deployment|containerapp)|serverless\s+deploy|pulumi\s+up|"
                       r"flyctl\s+deploy|vercel|netlify\s+deploy|docker/build-push-action|"
                       r"docker\s+push|environment:\s*(production|prod)\b", re.I)
TEST_STEP_RE = re.compile(r"pytest|tox\b|nox\b|npm\s+(run\s+)?test|yarn\s+(run\s+)?test|pnpm\s+(run\s+)?test|"
                          r"jest|vitest|mocha|go\s+test|cargo\s+(test|nextest)|mvn\s|gradle|dotnet\s+test|"
                          r"make\s+(test|check)|phpunit|rspec|ctest|playwright|cypress", re.I)
K8S_KIND_RE = re.compile(r"^kind:\s*(Deployment|StatefulSet|DaemonSet)\s*$", re.M)

# ---------- scoring ----------
def cap(x, full):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return np.nan
    return float(min(1.0, max(0.0, x / full)))

def mean_avail(xs):
    xs = [x for x in xs if x is not None and not (isinstance(x, float) and math.isnan(x))]
    return float(np.mean(xs)) if xs else np.nan

def to_level(score, cuts):
    """Map a 0-1 score to COCOMO levels coded VL=-2 ... XH=3."""
    if score is None or (isinstance(score, float) and math.isnan(score)):
        return np.nan
    return -2 + int(np.searchsorted(cuts, score, side="right"))

# ---------- tree-based measures at the end of a window ----------
def tree_measures(repo_dir, sha, max_yaml=300):
    files = list_files(repo_dir, sha)
    m = {}
    ci_files = [f for f in files if CI_RE.search(f)]
    m["has_ci"] = int(bool(ci_files))
    iac_files = [f for f in files if is_iac(f)]
    m["n_iac_files"] = len(iac_files)
    m["has_ai_config"] = int(any(AI_CFG_RE.search(f) for f in files))
    m["has_security_policy"] = int(any(SECPOL_RE.search(f) for f in files))
    m["has_dep_update"] = int(any(DEPUPD_RE.search(f) for f in files))
    m["has_codeowners"] = int(any(CODEOWNERS_RE.search(f) for f in files))

    # CI workflow contents: SAST and deployment steps
    ci_text = "\n".join(read_file(repo_dir, sha, f) for f in ci_files[:60])
    m["has_sast"] = int(bool(SAST_RE.search(ci_text)))
    m["ci_deploy"] = int(bool(DEPLOY_RE.search(ci_text)))
    m["has_test_in_ci"] = int(bool(TEST_STEP_RE.search(ci_text)))
    m["n_ci_workflows"] = len(ci_files)

    # services: docker-compose services with a build key
    compose_built = 0
    for f in [f for f in files if COMPOSE_RE.search(f) and not NONPROD_DIR_RE.search(f)][:20]:
        try:
            doc = yaml.safe_load(read_file(repo_dir, sha, f)) or {}
            svcs = doc.get("services", {}) if isinstance(doc, dict) else {}
            if isinstance(svcs, dict):
                compose_built = max(compose_built, sum(1 for v in svcs.values()
                                                       if isinstance(v, dict) and "build" in v))
        except Exception:
            pass
    # Kubernetes workloads in deployment directories
    # unique workload names, so the same service defined in Helm, Kustomize and plain manifests counts once
    names = set()
    yfiles = [f for f in files if f.lower().endswith((".yml", ".yaml")) and K8S_DIR_RE.search(f)
              and not NONPROD_DIR_RE.search(f)][:max_yaml]
    for f in yfiles:
        for doc in re.split(r"(?m)^---\s*$", read_file(repo_dir, sha, f)):
            if K8S_KIND_RE.search(doc):
                mm = re.search(r"(?m)^metadata:\s*\n(?:[ \t]+.*\n)*?[ \t]+name:\s*['\"]?([^\s'\"]+)", doc)
                nm = mm.group(1) if mm else f
                if "{{" in nm:                      # Helm template name: use the file name instead
                    nm = os.path.splitext(os.path.basename(f))[0]
                names.add(nm.lower())
    k8s = len(names)
    # directories with their own Dockerfile
    # a service = a folder with its own Dockerfile AND its own build file (not a packaging/dev image folder)
    build_dirs = {os.path.dirname(f) for f in files if BUILD_FILE_RE.search(f)}
    dock_dirs = {os.path.dirname(f) for f in files if DOCKERFILE_RE.search(f) and not NONPROD_DIR_RE.search(f)
                 and not PACKAGING_DIR_RE.search(os.path.dirname(f) + "/") and os.path.dirname(f) in build_dirs}
    m["svc_compose_built"], m["svc_k8s_workloads"], m["svc_dockerfile_dirs"] = compose_built, k8s, len(dock_dirs)
    m["n_services"] = max(1, compose_built, k8s, len(dock_dirs))
    m["_iac_set"] = set(iac_files)
    m["_ci_set"] = set(ci_files)
    return m

# ---------- one window ----------
def window_measures(commits, first_dates, tags, w_start, w_end, cfg):
    cw = [c for c in commits if w_start <= c["date"] < w_end]
    for c in cw:
        c["kind"] = author_kind(c["name"], c["email"])
    human = [c for c in cw if c["kind"] == "human"]
    row = {"commits_total": len(cw), "commits_human": len(human),
           "commits_bot": sum(c["kind"] == "bot" for c in cw),
           "commits_ai_author": sum(c["kind"] == "ai" for c in cw)}

    # effort: full-time-equivalent developer-months (Robles et al.)
    per_dev_month = defaultdict(int)
    for c in human:
        per_dev_month[(c["email"], c["date"].strftime("%Y-%m"))] += 1
    T = cfg["FT_COMMITS_PER_MONTH"]
    row["effort_pm"] = float(sum(min(1.0, n / T) for n in per_dev_month.values()))

    dev_commits = defaultdict(int)
    for c in human:
        dev_commits[c["email"]] += 1
    row["team_size"] = len(dev_commits)
    row["core_devs"] = sum(1 for n in dev_commits.values() if n >= cfg["CORE_DEV_COMMITS"])

    # size: source lines added (added + modified lines appear as added in numstat)
    # bulk commits (imports, generated or vendored code) are left out of size
    src = test = bulk = nbulk = 0
    for c in human:
        cs = sum(a for a, d, p in c["files"] if file_class(p) == "src")
        if cs > cfg["BULK_COMMIT_LINES"]:
            bulk += cs
            nbulk += 1
            continue
        src += cs
        test += sum(a for a, d, p in c["files"] if file_class(p) == "test")
    row["ksloc"] = src / 1000.0
    row["ksloc_test"] = test / 1000.0
    row["ksloc_bulk_excluded"] = bulk / 1000.0
    row["n_bulk_commits"] = nbulk

    # AI commits (authored by agents or with an AI co-author trailer)
    nonbot = [c for c in cw if c["kind"] != "bot"]
    ai_commits = sum(1 for c in nonbot if c["kind"] == "ai" or has_ai_coauthor(c["body"]))
    row["ai_commit_share"] = ai_commits / len(nonbot) if nonbot else np.nan

    # releases
    months = (w_end - w_start).days / 30.44
    row["releases_per_month"] = sum(w_start <= t < w_end for t in tags) / months

    # team experience: commit-weighted mean months since each developer's first commit
    tenures, weights = [], []
    for e, n in dev_commits.items():
        fd = first_dates.get(e)
        if fd is not None:
            tenures.append(max(0.0, (w_start - fd).days / 30.44))
            weights.append(n)
    row["tenure_months"] = float(np.average(tenures, weights=weights)) if tenures else np.nan

    # company-led signal: share of human commits from the most common corporate email domain
    PUBLIC = {"gmail.com", "users.noreply.github.com", "outlook.com", "hotmail.com", "yahoo.com",
              "qq.com", "163.com", "protonmail.com", "proton.me", "icloud.com", "me.com", "live.com",
              "googlemail.com", "gmx.de", "web.de", "mail.ru", "yandex.ru", "126.com", "foxmail.com"}
    doms = defaultdict(int)
    for c in human:
        d = c["email"].split("@")[-1]
        if d and d not in PUBLIC and "." in d:
            doms[d] += 1
    top = max(doms.items(), key=lambda kv: kv[1]) if doms else (None, 0)
    row["top_corp_domain"] = top[0]
    row["top_corp_domain_share"] = top[1] / len(human) if human else np.nan
    row["_human_commits"] = human
    return row

## 4. API helpers (GitHub, Scorecard, AIDev)

In [ ]:
# ===================== GITHUB / SCORECARD / AIDEV HELPERS =====================
import time, random, requests

SESSION = requests.Session()

def gh_get(url, params=None, search=False):
    """GET from the GitHub API with automatic waiting on rate limits."""
    if not url.startswith("http"):
        url = "https://api.github.com" + url
    headers = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json",
               "X-GitHub-Api-Version": "2022-11-28"}
    for attempt in range(6):
        if search:
            time.sleep(2.2)              # search API: 30 requests per minute
        r = SESSION.get(url, headers=headers, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (403, 429) and (r.headers.get("X-RateLimit-Remaining") == "0"
                                            or "rate limit" in r.text.lower()):
            reset = int(r.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait = max(5, reset - time.time() + 5)
            print(f"  rate limit reached, waiting {wait/60:.1f} min")
            time.sleep(wait)
            continue
        if r.status_code in (502, 503, 504):
            time.sleep(10 * (attempt + 1))
            continue
        if r.status_code in (404, 409, 410, 422, 451):
            return None
        time.sleep(5)
    return None

def commit_count_since(full_name, since):
    """Number of commits on the default branch since a date (one API call, via the Link header)."""
    url = f"https://api.github.com/repos/{full_name}/commits"
    headers = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json"}
    try:
        r = SESSION.get(url, headers=headers, params={"since": since, "per_page": 1}, timeout=60)
        if r.status_code != 200:
            return 0
        m = re.search(r'[?&]page=(\d+)>; rel="last"', r.headers.get("Link", ""))
        return int(m.group(1)) if m else len(r.json())
    except Exception:
        return 0

def repo_meta(full_name):
    j = gh_get(f"/repos/{full_name}")
    if not j:
        return None
    return {"full_name": j["full_name"], "stars": j["stargazers_count"], "language": j["language"],
            "created_at": j["created_at"], "fork": j["fork"], "archived": j["archived"],
            "size_mb": j["size"] / 1024, "owner_type": j["owner"]["type"],
            "default_branch": j["default_branch"]}

def merged_pr_count(full_name, a, b):
    q = f"repo:{full_name} is:pr is:merged merged:{a:%Y-%m-%d}..{(b - pd.Timedelta(days=1)):%Y-%m-%d}"
    j = gh_get("/search/issues", params={"q": q, "per_page": 1}, search=True)
    return j["total_count"] if j else np.nan

def ci_run_count(full_name, a, b):
    j = gh_get(f"/repos/{full_name}/actions/runs",
               params={"created": f"{a:%Y-%m-%d}..{(b - pd.Timedelta(days=1)):%Y-%m-%d}", "per_page": 1})
    return j.get("total_count", np.nan) if j else np.nan

SCORECARD_CHECKS = ["Branch-Protection", "Code-Review", "SAST", "Dependency-Update-Tool", "Security-Policy"]

def scorecard(full_name):
    """Current OpenSSF Scorecard check scores (0-10); NaN if not available."""
    try:
        r = SESSION.get(f"https://api.securityscorecards.dev/projects/github.com/{full_name}", timeout=60)
        if r.status_code != 200:
            return {}
        checks = {c["name"]: c.get("score", -1) for c in r.json().get("checks", [])}
        return {f"sc_{k}": (checks[k] if checks.get(k, -1) >= 0 else np.nan) for k in SCORECARD_CHECKS}
    except Exception:
        return {}

# ---------- AIDev ----------
def load_aidev(version="v5"):
    base = f"hf://datasets/hao-li/AIDev@{version}"
    prs = pd.read_parquet(f"{base}/all_pull_request.parquet",
                          columns=["repo_url", "agent", "created_at", "merged_at"])
    prs["repo"] = prs["repo_url"].str.extract(r"github\.com/(?:repos/)?([^/]+/[^/]+?)(?:\.git)?/?$")[0]
    prs["created_at"] = pd.to_datetime(prs["created_at"], utc=True, errors="coerce")
    prs["merged_at"] = pd.to_datetime(prs["merged_at"], utc=True, errors="coerce")
    try:
        repos = pd.read_parquet(f"{base}/all_repository.parquet")
    except Exception as e:
        print("Could not load all_repository:", e)
        repos = None
    return prs, repos

def detect_col(df, *keys):
    for k in keys:
        for c in df.columns:
            if k in c.lower():
                return c
    return None

In [ ]:
rl = gh_get('/rate_limit')
print('GitHub API OK, requests left this hour:', rl['resources']['core']['remaining'] if rl else 'TOKEN PROBLEM')

## 5. Per-repository processing

In [ ]:
# ===================== PROCESS ONE REPOSITORY =====================
def process_repo(full_name, group, cfg, aidev_prs, aidev_cutoff, use_api=True):
    out_csv = os.path.join(cfg["OUT_DIR"], "repos", full_name.replace("/", "__") + ".csv")
    if os.path.exists(out_csv):
        print(f"  already done: {full_name}")
        return pd.read_csv(out_csv)
    meta = repo_meta(full_name) if use_api else {}
    sc = scorecard(full_name) if use_api else {}
    tmp = os.path.join(cfg["CLONE_DIR"], full_name.replace("/", "__"))
    t0 = time.time()
    clone_repo(full_name, tmp)
    try:
        windows = make_windows(cfg["START"], cfg["END"], cfg["WINDOW_MONTHS"])
        commits = read_commits(tmp, cfg["START"], cfg["END"])
        firsts = first_commit_dates(tmp)
        tags = tag_dates(tmp)
        agent = aidev_prs[(aidev_prs["repo"] == full_name) & aidev_prs["merged_at"].notna()] \
            if aidev_prs is not None else None
        rows = []
        for label, a, b in windows:
            sha = commit_at(tmp, b)
            if sha is None:
                continue
            r = {"repo": full_name, "group": group, "window": label,
                 "window_start": a.date().isoformat(), "end_commit": sha}
            r.update(window_measures(commits, firsts, tags, a, b, cfg))
            human = r.pop("_human_commits")
            if r["commits_total"] == 0:
                continue
            t = tree_measures(tmp, sha)
            iac_set = t.pop("_iac_set")
            ci_set = t.pop("_ci_set")
            r.update(t)
            r["iac_change_share"] = (sum(1 for c in human if any(p in iac_set for _, _, p in c["files"]))
                                     / len(human)) if human else np.nan
            r["ci_change_share"] = (sum(1 for c in human if any(p in ci_set for _, _, p in c["files"]))
                                    / len(human)) if human else np.nan
            # API-based window measures
            if use_api:
                r["merged_prs"] = merged_pr_count(full_name, a, b)
                r["ci_runs"] = ci_run_count(full_name, a, b)
            else:
                r["merged_prs"] = r["ci_runs"] = np.nan
            weeks = (b - a).days / 7
            r["ci_runs_per_week"] = r["ci_runs"] / weeks if not pd.isna(r["ci_runs"]) else np.nan
            if agent is not None and a < aidev_cutoff:
                n_agent = int(((agent["merged_at"] >= a) & (agent["merged_at"] < b)).sum())
                r["agent_merged_prs"] = n_agent
                r["agent_pr_share"] = (n_agent / r["merged_prs"]
                                       if r["merged_prs"] and not pd.isna(r["merged_prs"]) else np.nan)
            else:
                r["agent_merged_prs"] = r["agent_pr_share"] = np.nan   # window after AIDev coverage ends
            r.update(score_window(r, cfg))
            r.update({k: v for k, v in (meta or {}).items() if k in ("stars", "language", "created_at",
                                                                     "owner_type", "size_mb")})
            r.update(sc)
            r["include"] = int(r["core_devs"] >= cfg["MIN_CORE_DEVS"] and r["ksloc"] >= cfg["MIN_KSLOC"])
            rows.append(r)
        df = pd.DataFrame(rows)
        os.makedirs(os.path.dirname(out_csv), exist_ok=True)
        df.to_csv(out_csv, index=False)
        print(f"  done {full_name}: {len(df)} windows, {df['include'].sum() if len(df) else 0} included, "
              f"{(time.time() - t0)/60:.1f} min")
        return df
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

def score_window(r, cfg):
    """Turn raw indicators into 0-1 driver scores and COCOMO levels."""
    f = cfg["FULL_SCORE_AT"]
    s = {}
    # CI run counts are NOT used: the GitHub API keeps run history for about one year only
    s["DVOP"] = mean_avail([r["has_ci"], r["has_test_in_ci"], cap(r["n_ci_workflows"], f["n_ci_workflows"]),
                            cap(r["releases_per_month"], f["releases_per_month"])])
    s["AIAD"] = mean_avail([cap(r["agent_pr_share"], f["agent_pr_share"]),
                            cap(r["ai_commit_share"], f["ai_commit_share"]), r["has_ai_config"]])
    s["MSVC"] = min(1.0, math.log(r["n_services"]) / math.log(f["n_services"]))
    s["CLAU"] = mean_avail([int(r["n_iac_files"] > 0), cap(r["iac_change_share"], f["iac_change_share"]),
                            r["ci_deploy"]])
    s["MTEX"] = cap(r["tenure_months"], f["tenure_months"])
    s["SECU"] = mean_avail([r["has_security_policy"], r["has_dep_update"], r["has_sast"], r["has_codeowners"]])
    for k in list(s):
        s[f"{k}_level"] = to_level(s[k], cfg["LEVEL_CUTS"])
    return s

## 6. Load AIDev
Downloads the pull-request table from Hugging Face (a few minutes the first time).

In [ ]:
aidev_prs, aidev_repos = load_aidev(CFG["AIDEV_VERSION"])
AIDEV_CUTOFF = aidev_prs["created_at"].max()
print(f"{len(aidev_prs):,} agent PRs in {aidev_prs['repo'].nunique():,} repositories; "
      f"data end: {AIDEV_CUTOFF:%Y-%m-%d}")
print("Agents:", aidev_prs["agent"].value_counts().to_dict())
if aidev_repos is not None:
    print("Repository table columns:", list(aidev_repos.columns))

## 7. Choose the pilot repositories
Picks 10 AI-agent repositories from AIDev and one matched comparison repository for each (same language, similar stars, no AIDev activity). The list is saved, so it is chosen only once.

To use your own list instead, set `MANUAL` to a list of `(repo, group)` pairs, e.g. `[("owner/name", "ai"), ("owner/other", "control")]`.

In [ ]:
# ===================== CHOOSE THE PILOT REPOSITORIES =====================
def eligible(m, cfg):
    ok = (m is not None and not m["fork"] and not m["archived"]
            and m["stars"] >= cfg["MIN_STARS"] and m["language"] in cfg["LANGS"]
            and m["created_at"] < cfg["CREATED_BEFORE"] and m["size_mb"] <= cfg["MAX_REPO_MB"])
    if not ok:
        return False
    # activity screen: enough recent commits to have a real team (saves cloning inactive repos)
    m["recent_commits"] = commit_count_since(m["full_name"], cfg["ACTIVE_SINCE"])
    return m["recent_commits"] >= cfg["MIN_RECENT_COMMITS"]

def select_ai_repos(prs, repos, cfg, n):
    merged = (prs[prs["merged_at"].notna() & prs["repo"].notna()]
              .groupby("repo").agg(n_agent_prs=("agent", "size"), first_agent_pr=("merged_at", "min"))
              .reset_index())
    cand = merged[(merged["n_agent_prs"] >= cfg["MIN_AGENT_PRS"]) &
                  (merged["first_agent_pr"] <= pd.Timestamp(cfg["ADOPTED_BY"], tz="UTC"))]
    # use AIDev's own star counts to pre-filter, if the column can be found
    if repos is not None:
        name_col = detect_col(repos, "full_name", "repo_url", "html_url", "url")
        star_col = detect_col(repos, "star")
        if name_col and star_col:
            rr = repos[[name_col, star_col]].copy()
            rr["repo"] = rr[name_col].astype(str).str.extract(
                r"(?:github\.com/(?:repos/)?)?([^/\s]+/[^/\s]+?)(?:\.git)?/?$")[0]
            tmp = cand.merge(rr[["repo", star_col]].rename(columns={star_col: "aidev_stars"}),
                             on="repo", how="left")
            if tmp["aidev_stars"].notna().mean() > 0.5:
                cand = tmp[tmp["aidev_stars"] >= cfg["MIN_STARS"]]
    print(f"AIDev repositories with >= {cfg['MIN_AGENT_PRS']} merged agent PRs, adopted by {cfg['ADOPTED_BY']}: {len(cand)}")
    cand = cand.sample(frac=1, random_state=cfg["SEED"])
    chosen, checked = [], 0
    for _, row in cand.iterrows():
        checked += 1
        m = repo_meta(row["repo"])
        if eligible(m, cfg):
            m.update(group="ai", first_agent_pr=row["first_agent_pr"], n_agent_prs=row["n_agent_prs"])
            chosen.append(m)
            print(f"  AI repo {len(chosen)}/{n}: {m['full_name']} ({m['language']}, {m['stars']} stars)")
        if len(chosen) >= n or checked >= cfg["MAX_CANDIDATES_CHECKED"]:
            break
    return chosen

def select_control_repos(ai_chosen, aidev_repos, cfg):
    """One comparison repository per AI repository: same language, similar stars, no AIDev activity."""
    used = set(aidev_repos) | {m["full_name"] for m in ai_chosen}
    chosen = []
    for m in ai_chosen:
        lo, hi = max(cfg["MIN_STARS"], m["stars"] // 2), m["stars"] * 2
        q = (f'language:"{m["language"]}" stars:{lo}..{hi} created:<{cfg["CREATED_BEFORE"]} '
             f'pushed:>{cfg["ACTIVE_AFTER"]} fork:false archived:false')
        j = gh_get("/search/repositories", params={"q": q, "sort": "updated", "per_page": 50}, search=True)
        for it in (j or {}).get("items", []):
            fn = it["full_name"]
            if fn in used:
                continue
            c = repo_meta(fn)
            if eligible(c, cfg):
                used.add(fn)
                c.update(group="control", matched_to=m["full_name"])
                chosen.append(c)
                print(f"  control for {m['full_name']}: {fn} ({c['stars']} stars)")
                break
    return chosen

In [ ]:
MANUAL = None
pilot_csv = CFG["OUT_DIR"] + "/pilot_repos.csv"
if MANUAL:
    pilot = pd.DataFrame(MANUAL, columns=["repo", "group"])
elif os.path.exists(pilot_csv):
    pilot = pd.read_csv(pilot_csv)
    print("Using saved pilot list")
else:
    ai = select_ai_repos(aidev_prs, aidev_repos, CFG, CFG["N_AI"])
    ctrl = select_control_repos(ai, aidev_prs["repo"].dropna().unique(), CFG)
    pilot = pd.DataFrame(ai + ctrl).rename(columns={"full_name": "repo"})
    pilot.to_csv(pilot_csv, index=False)
display(pilot[[c for c in ["repo", "group", "language", "stars", "created_at", "matched_to",
                           "n_agent_prs"] if c in pilot.columns]])

## 8. Run the pipeline
Each repository is cloned, measured, saved to Drive, and deleted from local disk.

In [ ]:
failures = []
for i, row in pilot.reset_index(drop=True).iterrows():
    print(f"[{i+1}/{len(pilot)}] {row['repo']} ({row['group']})")
    try:
        process_repo(row["repo"], row["group"], CFG, aidev_prs, AIDEV_CUTOFF)
    except Exception as e:
        print("  FAILED:", str(e)[:300])
        failures.append((row["repo"], str(e)[:300]))
print("Failures:", failures or "none")

## 9. Combine and summarise

In [ ]:
import glob
data = pd.concat([pd.read_csv(f) for f in glob.glob(CFG["OUT_DIR"] + "/repos/*.csv")], ignore_index=True)
# AI adoption = first window with ANY AI signal (agent PRs, AI co-authored commits, AI config file);
# comparison repositories that start using AI become 'adopters' too (handled by the staggered design)
sig = (data["agent_merged_prs"].fillna(0) > 0) | (data["ai_commit_share"].fillna(0) > 0) | (data["has_ai_config"] == 1)
first = data[sig].groupby("repo")["window_start"].min()
data["ai_adopt_start"] = data["repo"].map(first)
data["post_ai"] = (data["ai_adopt_start"].notna() & (data["window_start"] >= data["ai_adopt_start"])).astype(int)
data.to_csv(CFG["OUT_DIR"] + "/pilot_dataset.csv", index=False)
print("Repositories with an AI signal:", first.size, "| control repositories that adopted AI:",
      data[(data["group"] == "control") & data["ai_adopt_start"].notna()]["repo"].nunique())
print(f"{data['repo'].nunique()} repositories, {len(data)} windows, {int(data['include'].sum())} meet the inclusion rules")
drivers = ["DVOP", "AIAD", "MSVC", "CLAU", "MTEX", "SECU"]
display(data[data["include"] == 1][["effort_pm", "ksloc", "team_size"] + drivers].describe().round(2))
display(data.groupby("group")[["effort_pm", "ksloc"] + drivers].mean().round(2))
inc = data[data["include"] == 1]
print("Share of included windows at each level (VL=-2 ... XH=3):")
display(pd.concat({c: inc[f"{c}_level"].value_counts(normalize=True).sort_index() for c in drivers}, axis=1).round(2))
print("Data-based cut-offs (20/40/60/80/95th percentiles of each score) - candidates for LEVEL_CUTS in the pre-registration:")
display(inc[drivers].quantile([0.2, 0.4, 0.6, 0.8, 0.95]).round(3))

## 10. Check the measures by hand
Pick a repository and window, open the links, and compare with the numbers. Record any mismatch and fix the rule in Section 3 **before** the full run.

What to check: (1) CI files exist and match `has_ci`; (2) the number of services matches the docker-compose / Kubernetes files; (3) IaC files match `n_iac_files`; (4) bots are not counted as human developers; (5) AI config files and AI co-authored commits are real; (6) SECURITY.md, Dependabot/Renovate, CODEOWNERS and SAST in CI match.

In [ ]:
CHECK_REPO = pilot["repo"].iloc[0]          # change to any pilot repository
CHECK_WINDOW = None                          # e.g. "2025H1"; None = latest window
d = data[data["repo"] == CHECK_REPO].sort_values("window")
r = d.iloc[-1] if CHECK_WINDOW is None else d[d["window"] == CHECK_WINDOW].iloc[0]
a = pd.Timestamp(r["window_start"]); b = a + pd.DateOffset(months=CFG["WINDOW_MONTHS"])
base = f"https://github.com/{CHECK_REPO}"
print("Files at window end:  ", f"{base}/tree/{r['end_commit']}")
print("CI workflows:         ", f"{base}/tree/{r['end_commit']}/.github/workflows")
print("Commits in window:    ", f"{base}/commits?since={a:%Y-%m-%d}&until={(b - pd.Timedelta(days=1)):%Y-%m-%d}")
print("Merged PRs in window: ", f"{base}/pulls?q=is%3Apr+is%3Amerged+merged%3A{a:%Y-%m-%d}..{(b - pd.Timedelta(days=1)):%Y-%m-%d}")
cols = ["window", "commits_human", "commits_bot", "commits_ai_author", "team_size", "core_devs",
        "effort_pm", "ksloc", "ksloc_bulk_excluded", "n_bulk_commits", "has_ci", "has_test_in_ci", "n_ci_workflows", "releases_per_month", "agent_pr_share",
        "ai_commit_share", "has_ai_config", "n_services", "svc_compose_built", "svc_k8s_workloads",
        "svc_dockerfile_dirs", "n_iac_files", "ci_deploy", "tenure_months", "has_security_policy",
        "has_dep_update", "has_sast", "has_codeowners"] + drivers
display(r[cols].to_frame().T)

## 11. Preview analysis (pilot only)
A quick check that the data behave sensibly: driver correlations, multicollinearity, and a simple COCOMO-style regression. **Do not report these numbers in the paper**: the pilot is too small, and the real analysis is the Bayesian model on the full sample.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
d = data[(data["include"] == 1) & (data["effort_pm"] > 0) & (data["ksloc"] > 0)].copy()
d["ln_pm"], d["ln_ksloc"] = np.log(d["effort_pm"]), np.log(d["ksloc"])
print("Driver correlations:"); display(d[drivers].corr().round(2))
X = d[drivers].dropna()
X = X.loc[:, X.std() > 0]
if len(X) > len(X.columns) + 2:
    Xc = np.column_stack([np.ones(len(X)), X.values])
    print("VIF:", {c: round(variance_inflation_factor(Xc, i + 1), 2) for i, c in enumerate(X.columns)})
levels = [f"{c}_level" for c in X.columns]
try:
    if len(d) < 30:
        raise ValueError(f"only {len(d)} included windows; need at least 30 for a meaningful preview")
    m = smf.ols("ln_pm ~ ln_ksloc + " + " + ".join(levels), data=d).fit(
        cov_type="cluster", cov_kwds={"groups": d["repo"]})
    print(m.summary().tables[1])
    print("Size exponent B (preview):", round(m.params["ln_ksloc"], 3), "- plausible range is about 0.9 to 1.2")
except Exception as e:
    print("Regression not possible yet:", e)
sc_cols = [c for c in d.columns if c.startswith("sc_")]
if sc_cols:
    d["scorecard_mean"] = d[sc_cols].mean(axis=1) / 10
    print("SECU vs current OpenSSF Scorecard, correlation:",
          round(d[["SECU", "scorecard_mean"]].corr().iloc[0, 1], 2))

## 12. Download the results (needed if you are not using Google Drive)

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/cocomo_pilot_results", "zip", CFG["OUT_DIR"])
files.download(zip_path)